# Phase 0b — vLLM Hook Spike

**Goal:** verify PyTorch forward hooks installed inside a vLLM Worker via the `WorkerExtension` API fire reliably when `llm.generate()` is called, **with vLLM's full V1 engine running** (`torch.compile` + CUDA graphs on, no `enforce_eager`).

**Go/no-go gate** for the TransformerLens vLLM source plan.

## How to run

1. **Runtime → Change runtime type → T4 GPU** (or better).
2. **Runtime → Run all.** Stop and paste output to Claude on the first cell that errors.
3. Cells **A and B are diagnostic controls** (bare vLLM, no extension). If they fail, the environment is broken and the spike can't even start; we'll fix that before touching the extension.
4. Cells **C onward** test the actual extension hypothesis.

In [ ]:
!pip install -q vllm transformers
import vllm, torch
print("vllm:", vllm.__version__)
print("torch:", torch.__version__, "cuda:", torch.cuda.is_available(),
      "device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")

## Control A — bare vLLM construction (no extension)

Verifies vLLM constructs on this Colab at all. **If this fails, the issue is environmental** (vLLM/CUDA/model-download), not our extension. Paste error + stop.

In [ ]:
import os
os.environ["VLLM_LOGGING_LEVEL"] = "DEBUG"
# Force engine core to run in-process (no separate subprocess for the scheduler) so we
# can actually SEE its traceback on failure instead of getting an empty 'Failed core proc(s): {}'.
os.environ["VLLM_USE_V1"] = "1"
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"

from vllm import LLM, SamplingParams
from vllm.inputs import TokensPrompt

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

print(f"Control A: constructing bare LLM for {MODEL_NAME} (no extension)...")
llm_control = LLM(
    model=MODEL_NAME,
    tensor_parallel_size=1,
    skip_tokenizer_init=True,
    gpu_memory_utilization=0.5,
    disable_log_stats=True,
    dtype="bfloat16",
)
print("Control A: LLM constructed.")

In [ ]:
# Control A continued — generate once to confirm full path works.
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
PROMPT = "The quick brown fox jumps over the"
ids = tok(PROMPT, add_special_tokens=True).input_ids
out = llm_control.generate(
    prompts=[TokensPrompt(prompt_token_ids=ids)],
    sampling_params=SamplingParams(max_tokens=1, temperature=0.0),
)
print("Control A: generated token:", out[0].outputs[0].token_ids)
print("Control A: PASS — vLLM works on this Colab without our extension.")

# Release control LLM to free GPU memory for the next test.
del llm_control
import gc, torch
gc.collect()
torch.cuda.empty_cache()
print("Control A: cleaned up.")

## Control B — vLLM construction WITH empty WorkerExtension

Uses a class with no overrides. **If this fails but Control A passed, the `worker_extension_cls` kwarg itself is broken on this vLLM version.** If it passes, we know the API works and the issue is something specific to our extension.

In [ ]:
%%writefile /content/tl_empty_ext.py
class TLEmptyExtension:
    pass

In [ ]:
import os, sys, shutil, site
os.environ["PYTHONPATH"] = "/content" + (os.pathsep + os.environ["PYTHONPATH"] if os.environ.get("PYTHONPATH") else "")
sys.path.insert(0, "/content")
_site = site.getsitepackages()[0]
shutil.copy("/content/tl_empty_ext.py", _site)

# Verify importable from this kernel.
if "tl_empty_ext" in sys.modules: del sys.modules["tl_empty_ext"]
import tl_empty_ext
assert hasattr(tl_empty_ext, "TLEmptyExtension")
print(f"Verified: tl_empty_ext.TLEmptyExtension at {tl_empty_ext.__file__}")

print(f"\nControl B: constructing LLM with empty WorkerExtension...")
llm_b = LLM(
    model=MODEL_NAME,
    tensor_parallel_size=1,
    skip_tokenizer_init=True,
    gpu_memory_utilization=0.5,
    disable_log_stats=True,
    dtype="bfloat16",
    worker_extension_cls="tl_empty_ext.TLEmptyExtension",
)
print("Control B: LLM constructed with extension.")

# Try to RPC into the (empty) extension. Any method we call should fail with AttributeError
# IF the extension is actually mixed in. If we get a different error (like 'method not found'
# without traversing the extension), the mixin isn't happening.
try:
    llm_b.collective_rpc("nonexistent_method_for_testing")
    print("Control B: collective_rpc accepted (unexpectedly). Extension may not be mixed in.")
except AttributeError as e:
    print(f"Control B: AttributeError as expected — extension IS being mixed in: {e}")
except Exception as e:
    print(f"Control B: unexpected {type(e).__name__}: {e}")

del llm_b
import gc, torch; gc.collect(); torch.cuda.empty_cache()
print("Control B: cleaned up.")

## C — full TLSpikeExtension construction

Only proceed if Controls A and B passed. This is the real spike.

In [ ]:
%%writefile /content/tl_spike_ext.py
"""Phase 0b spike — minimal vLLM WorkerExtension.
Registers a single PyTorch forward hook on the module at a dot-path inside
the worker's loaded model, stashes outputs into a per-worker buffer.
"""
from typing import List
import torch


class TLSpikeExtension:
    """Mixed into vLLM's Worker. Methods exposed via collective_rpc."""

    def tl_install_capture(self, dot_path: str) -> str:
        model = self.model_runner.model
        target = model
        for seg in dot_path.split("."):
            target = target[int(seg)] if seg.isdigit() else getattr(target, seg)

        if not hasattr(self, "_tl_captures"):
            self._tl_captures: List[torch.Tensor] = []
            self._tl_fire_count: int = 0

        @torch.no_grad()
        def _hook(_module, _inputs, output):
            tensor = output[0] if isinstance(output, tuple) else output
            if isinstance(tensor, torch.Tensor):
                torch.cuda.current_stream().synchronize()
                self._tl_captures.append(tensor.detach().clone().cpu())
                self._tl_fire_count += 1

        target.register_forward_hook(_hook)
        return f"hook installed on {dot_path}; module: {type(target).__name__}"

    def tl_read_captures(self) -> List[torch.Tensor]:
        out = getattr(self, "_tl_captures", [])
        self._tl_captures = []
        return out

    def tl_read_fire_count(self) -> int:
        return getattr(self, "_tl_fire_count", 0)

    def tl_reset(self) -> None:
        self._tl_captures = []
        self._tl_fire_count = 0

In [ ]:
import os, sys, shutil, site
_site = site.getsitepackages()[0]
shutil.copy("/content/tl_spike_ext.py", _site)
if "tl_spike_ext" in sys.modules: del sys.modules["tl_spike_ext"]
import tl_spike_ext
assert hasattr(tl_spike_ext, "TLSpikeExtension")
print(f"Verified: tl_spike_ext.TLSpikeExtension at {tl_spike_ext.__file__}")

TARGET_DOT_PATH = "model.layers.0.self_attn"

print(f"\nC: constructing LLM with TLSpikeExtension...")
llm = LLM(
    model=MODEL_NAME,
    tensor_parallel_size=1,
    skip_tokenizer_init=True,
    gpu_memory_utilization=0.5,
    disable_log_stats=True,
    dtype="bfloat16",
    worker_extension_cls="tl_spike_ext.TLSpikeExtension",
)
print("C: LLM constructed.")

In [ ]:
# Install the capture hook BEFORE the first generate() call.
install_result = llm.collective_rpc("tl_install_capture", args=(TARGET_DOT_PATH,))
print("install_capture:", install_result)

out = llm.generate(
    prompts=[TokensPrompt(prompt_token_ids=ids)],
    sampling_params=SamplingParams(max_tokens=1, temperature=0.0),
)
print("generated token:", out[0].outputs[0].token_ids)

caps_per_worker = llm.collective_rpc("tl_read_captures")
fire_per_worker = llm.collective_rpc("tl_read_fire_count")
print(f"\nHook fire count per worker: {fire_per_worker}")
print(f"Capture list length per worker: {[len(c) for c in caps_per_worker]}")
if caps_per_worker and caps_per_worker[0]:
    print(f"First capture shape: {caps_per_worker[0][0].shape}, dtype: {caps_per_worker[0][0].dtype}")
    print(f"First capture norm: {caps_per_worker[0][0].float().norm().item():.4f}")

In [ ]:
# Determinism check — 100 back-to-back generates.
import time
llm.collective_rpc("tl_reset")
t0 = time.time()
for _ in range(100):
    llm.generate(
        prompts=[TokensPrompt(prompt_token_ids=ids)],
        sampling_params=SamplingParams(max_tokens=1, temperature=0.0),
    )
elapsed = time.time() - t0

fire_count = llm.collective_rpc("tl_read_fire_count")[0]
caps = llm.collective_rpc("tl_read_captures")[0]
print(f"100 generations in {elapsed:.1f}s  ({elapsed*10:.0f} ms/gen)")
print(f"Hook fire count: {fire_count}  (expected: 100)")
print(f"Captures collected: {len(caps)}")

if len(caps) >= 2:
    ref = caps[0]
    all_equal = all(torch.equal(ref, c) for c in caps[1:])
    print(f"All captures bitwise identical: {all_equal}")
    if not all_equal:
        first_bad = next((i+1 for i, c in enumerate(caps[1:]) if not torch.equal(ref, c)), None)
        if first_bad is not None:
            delta = (ref - caps[first_bad]).float().abs().max().item()
            print(f"First mismatching capture at idx {first_bad}; max abs delta: {delta:.6f}")

In [ ]:
# HF reference forward — sanity check the hook caught real attention output.
import torch
from transformers import AutoModelForCausalLM

hf_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16, device_map="cuda"
).eval()

hf_caps = []
def hf_hook(_m, _i, output):
    t = output[0] if isinstance(output, tuple) else output
    if isinstance(t, torch.Tensor):
        hf_caps.append(t.detach().clone().cpu())

target = hf_model
for seg in TARGET_DOT_PATH.split("."):
    target = target[int(seg)] if seg.isdigit() else getattr(target, seg)
handle = target.register_forward_hook(hf_hook)

with torch.no_grad():
    hf_model(torch.tensor([ids], device="cuda"))
handle.remove()

hf_tensor = hf_caps[0]
vllm_tensor = caps[0]
print(f"HF   tensor: shape={tuple(hf_tensor.shape)}  norm={hf_tensor.float().norm().item():.4f}")
print(f"vLLM tensor: shape={tuple(vllm_tensor.shape)}  norm={vllm_tensor.float().norm().item():.4f}")
ratio = vllm_tensor.float().norm().item() / hf_tensor.float().norm().item()
print(f"Norm ratio (vLLM / HF): {ratio:.3f}")

In [ ]:
# Phase 0b Summary — paste this back to Claude
import vllm, torch
print("=" * 60)
print("PHASE 0b SUMMARY")
print("=" * 60)
print(f"vllm:           {vllm.__version__}")
print(f"torch:          {torch.__version__}")
print(f"GPU:            {torch.cuda.get_device_name(0)}")
print(f"Model:          {MODEL_NAME}")
print(f"Target path:    {TARGET_DOT_PATH}")
print()
print(f"Fire count:     {fire_count} / 100   {'PASS' if fire_count == 100 else 'FAIL'}")
if len(caps) >= 2:
    all_equal = all(torch.equal(caps[0], c) for c in caps[1:])
    print(f"Bitwise det.:   {all_equal}    {'PASS' if all_equal else 'FAIL'}")
print(f"Norm ratio:     {ratio:.3f}  {'PASS' if 0.5 <= ratio <= 2.0 else 'INVESTIGATE'}")
print(f"Latency:        {elapsed*10:.0f} ms/gen ({elapsed:.1f}s for 100 runs)")
print("=" * 60)